# 청크·SearchResult 계약 검증

청크 metadata 필수 필드와 Retriever 출력 규격이 Data, Retrieval, Generation 공통 계약을 만족하는지 확인합니다.

In [1]:
from copy import deepcopy
from dataclasses import fields
from pathlib import Path
import sys
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser_chunker import load_chunks_jsonl, validate_chunk_contract
from src.retriever import SearchResult, SimpleRetriever

with (PROJECT_ROOT / "config" / "default.yaml").open(encoding="utf-8") as f:
    config = yaml.safe_load(f)

CHUNKS_PATH = Path(config["paths"]["chunks"])
SAMPLE_CHUNKS_PATH = PROJECT_ROOT / "samples" / "processed" / "sample_chunks.jsonl"
REQUIRED_METADATA_FIELDS = ("title", "project_name", "agency", "file_name")
EXPECTED_RESULT_FIELDS = {"chunk_id", "doc_id", "text", "metadata", "score"}

In [2]:
actual_chunks = load_chunks_jsonl(CHUNKS_PATH, validate=True)
print(f"실제 청크 검증 성공: {len(actual_chunks):,}개")
print("필수 metadata 필드:", ", ".join(REQUIRED_METADATA_FIELDS))

실제 청크 검증 성공: 10,222개
필수 metadata 필드: title, project_name, agency, file_name


In [3]:
sample_chunks = load_chunks_jsonl(SAMPLE_CHUNKS_PATH, validate=True)
results = SimpleRetriever(sample_chunks).search("제안서 제출", top_k=1)
result_fields = {field.name for field in fields(SearchResult)}

assert results
assert result_fields == EXPECTED_RESULT_FIELDS
print(f"샘플 청크 검증 성공: {len(sample_chunks)}개")
print("SearchResult 필드:", ", ".join(sorted(result_fields)))

샘플 청크 검증 성공: 3개
SearchResult 필드: chunk_id, doc_id, metadata, score, text


In [4]:
invalid_chunks = deepcopy(sample_chunks)
invalid_chunks[0]["metadata"].pop("agency")

try:
    validate_chunk_contract(invalid_chunks)
except ValueError as error:
    print(f"예상한 검증 오류: {error}")
else:
    raise AssertionError("필수 metadata 필드 누락을 감지하지 못했습니다.")

예상한 검증 오류: 1번째 청크 metadata에 필수 필드가 없습니다: agency
